<div align="center">
  <img src="https://raw.githubusercontent.com/eightmm/FoldJAX/main/docs/banner-light.png" width="760" alt="FoldJAX">
  <h2>One input, multiple JAX structure models</h2>
  <p>Build one FoldJAX job, run several public models, and compare every result in one notebook.</p>
</div>

### How to run

1. Choose **Runtime → Change runtime type → GPU** and a **Python 3.13** runtime.
2. Edit the form below. The default is a two-chain insulin example compared with **Protenix + OpenDDE**.
3. Choose **Runtime → Run all**. The dependency cell restarts the runtime once; reconnect and choose **Run all** one more time.
4. Review the model table before the downloads begin. Prediction runs are deliberately sequential so several large checkpoints are not resident on the GPU at once.

> **Scientific and privacy scope:** Fast demo mode uses 1 sample, 20 diffusion steps, and 1 recycle. It is a smoke/iteration schedule, not any model's released schedule. Native confidence fields are shown side by side but are not automatically ranked because names and calibration differ by model. MSA policy none does not contact an additional MSA service. Policy auto sends protein sequences to the public ColabFold MMseqs2 service. Colab, Google Drive, saved notebook outputs, and downloaded archives can still retain your input.


In [ ]:
# @title 1. Configure one input and the models to compare
import re

JOB_NAME = "insulin-complex"  # @param {type:"string"}
PROTEIN_CHAINS = "GIVEQCCTSICSLYQLENYCN:FVNQHLCGSHLVEALYLVCGERGFFYTPKT"  # @param {type:"string"}  # noqa: E501
MODEL_PRESET = "Compare: Protenix + OpenDDE"  # @param ["Quick: Protenix only", "Compare: Protenix + OpenDDE", "Public trio: Protenix + OpenDDE + Boltz-2", "Custom public models"]  # noqa: E501
CUSTOM_MODELS = "protenix,opendde"  # @param {type:"string"}
RUN_MODE = "Fast demo"  # @param ["Fast demo", "Released defaults"]
NUM_SEEDS = 1  # @param {type:"integer", min:1, max:5}
MSA_POLICY = "none"  # @param ["none", "auto", "required"]
CONTINUE_ON_ERROR = True  # @param {type:"boolean"}
PERSIST_WEIGHTS_TO_DRIVE = False  # @param {type:"boolean"}
DOWNLOAD_RESULTS = True  # @param {type:"boolean"}

MODEL_PRESETS = {
    "Quick: Protenix only": ("protenix",),
    "Compare: Protenix + OpenDDE": ("protenix", "opendde"),
    "Public trio: Protenix + OpenDDE + Boltz-2": (
        "protenix",
        "opendde",
        "boltz2",
    ),
}
PUBLIC_COLAB_MODELS = {"protenix", "opendde", "boltz2"}

if MODEL_PRESET == "Custom public models":
    SELECTED_MODELS = tuple(
        dict.fromkeys(
            part.strip().lower()
            for part in CUSTOM_MODELS.split(",")
            if part.strip()
        )
    )
else:
    SELECTED_MODELS = MODEL_PRESETS[MODEL_PRESET]

unknown_models = set(SELECTED_MODELS) - PUBLIC_COLAB_MODELS
if not SELECTED_MODELS or unknown_models:
    raise ValueError(
        "Choose one or more public Colab models from "
        f"{sorted(PUBLIC_COLAB_MODELS)}; invalid: {sorted(unknown_models)}"
    )
if not 1 <= int(NUM_SEEDS) <= 5:
    raise ValueError("NUM_SEEDS must be between 1 and 5.")

clean_input = re.sub(r"\s+", "", PROTEIN_CHAINS).upper()
PROTEIN_SEQUENCES = tuple(part for part in clean_input.split(":") if part)
if not PROTEIN_SEQUENCES or any(not chain.isalpha() for chain in PROTEIN_SEQUENCES):
    raise ValueError(
        "PROTEIN_CHAINS must contain letter-only protein chains separated by ':'."
    )
JOB_SLUG = re.sub(r"[^a-z0-9._-]+", "-", JOB_NAME.lower()).strip("-")
if not JOB_SLUG:
    raise ValueError("JOB_NAME must contain at least one letter or digit.")

RUN_LABEL = "fast-demo" if RUN_MODE == "Fast demo" else "released"
FAST_SCHEDULE = (
    {"num_samples": 1, "num_steps": 20, "num_recycles": 1}
    if RUN_MODE == "Fast demo"
    else {}
)
print(
    f"Job: {JOB_SLUG} | chains: {len(PROTEIN_SEQUENCES)} | "
    f"residues: {sum(map(len, PROTEIN_SEQUENCES))}"
)
print(f"Models: {', '.join(SELECTED_MODELS)} | mode: {RUN_MODE}")


In [ ]:
# @title 2. Install the pinned FoldJAX CUDA 12 stack
import os
import shutil
import signal
import subprocess
import sys
from pathlib import Path

FOLDJAX_REF = "613c0fa99b3838db1a9ea02db35735056ce71e96"
install_marker = Path(f"/content/.foldjax-colab-{FOLDJAX_REF}.installed")

if sys.version_info[:2] != (3, 13):
    install_marker.unlink(missing_ok=True)
    raise RuntimeError(
        f"FoldJAX requires Python 3.13; this runtime has "
        f"{sys.version.split()[0]}. In Colab, choose Runtime → Change runtime "
        "type → Runtime version, select a Python 3.13 GPU runtime, reconnect, "
        "and rerun this notebook from the first cell. If Python 3.13 is not "
        "offered, this notebook cannot run in that session."
    )
if shutil.which("nvidia-smi") is None:
    raise RuntimeError(
        "No NVIDIA GPU runtime was found. Select Runtime → Change runtime "
        "type → GPU and reconnect before installing."
    )

if not install_marker.exists():
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--upgrade",
            (
                "foldjax[cuda12] @ git+https://github.com/eightmm/"
                f"FoldJAX.git@{FOLDJAX_REF}"
            ),
            "py3Dmol>=2.0,<3",
        ]
    )
    install_marker.write_text(FOLDJAX_REF)
    print("Dependencies installed; restarting the runtime once...")
    os.kill(os.getpid(), signal.SIGKILL)
else:
    print(f"FoldJAX dependencies are installed from {FOLDJAX_REF[:12]}.")


### Runtime and persistent weights

Local Colab storage is fastest. Enable Drive persistence before **Run all** if you want downloaded and converted checkpoints to survive a runtime reset. Compilation entries stay local because they are tied to the JAX version, GPU, options, weights, and concrete input shape.


In [ ]:
# @title 3. Configure storage and portable kernels
WORK_DIR = Path("/content/foldjax-colab")
OUTPUT_ROOT = WORK_DIR / "outputs" / JOB_SLUG / RUN_LABEL
COMPILE_CACHE = WORK_DIR / "compile-cache"

if PERSIST_WEIGHTS_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    foldjax_home = Path("/content/drive/MyDrive/foldjax-cache")
else:
    foldjax_home = Path("/content/foldjax-cache")

os.environ["FOLDJAX_HOME"] = str(foldjax_home)
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.90")
os.environ["PROTENIX_TRIANGLE_MULTIPLICATION_BACKEND"] = "xla"
os.environ["PROTENIX_TRIANGLE_BACKEND"] = "xla_jit"
os.environ["BOLTZ_JAX_TRIANGLE_MULTIPLICATION_BACKEND"] = "xla"
for directory in (WORK_DIR, OUTPUT_ROOT, COMPILE_CACHE, foldjax_home):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Weight store: {foldjax_home}")
print(f"Outputs: {OUTPUT_ROOT}")


In [ ]:
# @title 4. Verify Python, JAX, CUDA packages, and the GPU
from importlib import metadata

import jax

expected_versions = {
    "cuequivariance": "0.11.1",
    "cuequivariance-jax": "0.11.1",
    "cuequivariance-ops-cu12": "0.11.1",
    "cuequivariance-ops-jax-cu12": "0.11.1",
    "flax": "0.12.9",
    "jax-cuda12-pjrt": "0.11.1",
    "jax-cuda12-plugin": "0.11.1",
    "jaxlib": "0.11.1",
    "qwix": "0.1.8",
    "tokamax": "0.0.13",
    "triton": "3.7.1",
}
installed_versions = {}
for package in expected_versions:
    try:
        installed_versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed_versions[package] = "missing"
version_mismatches = {
    package: (installed_versions[package], expected)
    for package, expected in expected_versions.items()
    if installed_versions[package] != expected
}
if jax.__version__ != "0.11.1" or version_mismatches:
    install_marker.unlink(missing_ok=True)
    raise RuntimeError(
        f"Expected JAX 0.11.1 and the pinned accelerator stack, but got "
        f"JAX {jax.__version__} with mismatches {version_mismatches}. "
        "Rerun the install cell; it will reinstall the stack and restart once."
    )

all_devices = jax.devices()
gpu_devices = [device for device in all_devices if device.platform == "gpu"]
gpu_info = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,driver_version",
        "--format=csv,noheader",
    ],
    check=False,
    capture_output=True,
    text=True,
)
print(f"Python {sys.version.split()[0]} | JAX {jax.__version__}")
print(f"JAX devices: {all_devices}")
if gpu_info.stdout.strip():
    print(gpu_info.stdout.strip())
if not gpu_devices:
    raise RuntimeError(
        "No JAX GPU was detected. Select Runtime → Change runtime type → GPU, "
        "restart, and run the notebook from the first cell."
    )


### Public model checkpoints

The next cell shows the publisher, licence, registered download size, and cache state for every selected model before downloading anything. It then fetches and verifies each bundle **sequentially**. Re-running skips verified assets. FoldJAX does not redistribute these checkpoints.

The disk check is deliberately conservative because source archives and converted JAX files can coexist during first setup. The public trio is intended for a roomy paid runtime or Drive-backed store; start with the two-model preset on a typical session.


In [ ]:
# @title 5. Inspect and fetch the selected model weights
import pandas as pd
from IPython.display import display

from foldjax import model_info

model_rows = []
model_infos = {}
pending_download_bytes = 0
for model_name in SELECTED_MODELS:
    info = model_info(model_name)
    model_infos[model_name] = info
    if not info.weights_fetchable:
        raise RuntimeError(
            f"{model_name} has no public managed download for this notebook."
        )
    if not info.weights_ready and info.download_bytes is not None:
        pending_download_bytes += info.download_bytes
    model_rows.append(
        {
            "model": info.model,
            "ready": info.weights_ready,
            "download_GB": (
                round(info.download_bytes / 1e9, 2)
                if info.download_bytes is not None
                else None
            ),
            "licence": info.weights_licence,
            "source": info.weights_source,
        }
    )

display(pd.DataFrame(model_rows))
free_bytes = shutil.disk_usage(foldjax_home).free
required_bytes = max(
    8_000_000_000,
    int(pending_download_bytes * 1.8 + 5_000_000_000),
)
print(
    f"Free storage: {free_bytes / 1e9:.1f} GB | "
    f"conservative setup target: {required_bytes / 1e9:.1f} GB"
)
if free_bytes < required_bytes:
    raise RuntimeError(
        "There is not enough free storage for the selected uncached bundles "
        "and conversion headroom. Select fewer models or enable Drive storage."
    )

for model_name in SELECTED_MODELS:
    print(f"\nPreparing {model_name}...")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "foldjax.cli",
            "weights",
            "fetch",
            "--model",
            model_name,
        ],
        check=True,
    )
print("All selected model bundles are ready.")


### One common input

Protein chains are separated with a colon in the form. Repeating a sequence creates a homomer; different sequences create a heteromer. FoldJAX converts this once into its common job schema and sends the same recorded input to every selected backend.

With MSA policy none, no additional MSA service is contacted. With auto or required, protein sequences are sent to the public ColabFold MMseqs2 service and the reusable alignment cache is shared across model runs. The notebook prints counts, not the sequence, so a saved output cell does not echo private input text.


In [ ]:
# @title 6. Build the common FoldJAX job
from foldjax import Job

job = Job.from_sequences(protein=PROTEIN_SEQUENCES, name=JOB_SLUG)
job_path = job.write(WORK_DIR / "input" / f"{JOB_SLUG}.json")
print(f"Input: {job_path}")
print(
    f"{len(PROTEIN_SEQUENCES)} protein chain(s), "
    f"{sum(map(len, PROTEIN_SEQUENCES))} total residues"
)


### Plan, then predict

All selected models receive one plural request. FoldJAX expands it into model-specific output namespaces and runs models sequentially, releasing one backend session before opening the next. Portable float32 and XLA attention are used because common Colab GPU types vary. Resume mode validates finished manifests and avoids repeating completed work.

If **Continue on error** is enabled, an out-of-memory or model-specific setup failure is recorded while later models still run. Programming defects and user cancellation still stop immediately.


In [ ]:
# @title 7. Resolve and review every model run
from foldjax import PredictionRequest, resolve_requests

batch_request = PredictionRequest(
    models=SELECTED_MODELS,
    input=job_path,
    output_dir=OUTPUT_ROOT,
    cache_dir=COMPILE_CACHE,
    seed=0,
    num_seeds=int(NUM_SEEDS),
    msa=MSA_POLICY,
    options={
        "dtype": "float32",
        "attention_kernel": "xla",
    },
    resume=True,
    on_error="continue" if CONTINUE_ON_ERROR else "stop",
    **FAST_SCHEDULE,
)
resolved_runs = resolve_requests(batch_request)
plan_rows = [
    {
        "model": run.model,
        "input": Path(run.input).name,
        "output": str(run.output_dir),
        "seeds": list(run.resolved_seeds),
        "schedule": run.sampling or "released defaults",
    }
    for run in resolved_runs
]
display(pd.DataFrame(plan_rows))


In [ ]:
# @title 8. Run all selected models
import json

from foldjax import predict_batch, progress

progress.enable()
report = predict_batch(batch_request)
print(
    f"Completed results: {len(report.results)} | "
    f"failures: {len(report.failures)} | resumed: {len(report.skipped)}"
)
if report.failures:
    display(pd.DataFrame(failure.summary() for failure in report.failures))


### Compare outputs without inventing a winner

The table keeps each model's native score names. A similarly named metric can still differ in calibration, input representation, and sampling schedule, so the notebook does not combine them into a universal ranking. Use the table to inspect consistency and uncertainty, and use the interactive viewer for structural comparison.


In [ ]:
# @title 9. Validate structures and build the comparison table
import math

import gemmi

comparison_rows = []
STRUCTURES = {}
for result in report.results:
    for sample_index, sample in enumerate(result.samples, start=1):
        if sample.structure_path is None:
            raise RuntimeError(f"{result.model} returned no structure path.")
        structure_path = Path(sample.structure_path)
        if not structure_path.is_file() or structure_path.stat().st_size == 0:
            raise RuntimeError(f"Missing or empty structure: {structure_path}")
        if structure_path.suffix.lower() in {".cif", ".mmcif"}:
            gemmi.cif.read_file(str(structure_path))
        else:
            gemmi.read_structure(str(structure_path))
        if not sample.scores or not all(
            math.isfinite(float(value)) for value in sample.scores.values()
        ):
            raise RuntimeError(
                f"{result.model} returned invalid confidence scores: "
                f"{sample.scores}"
            )

        label = f"{result.model} · seed {sample.seed} · sample {sample_index}"
        STRUCTURES[label] = structure_path
        row = {
            "model": result.model,
            "seed": sample.seed,
            "sample": sample_index,
            "structure": structure_path.name,
        }
        row.update(
            {
                f"score:{name}": float(value)
                for name, value in sorted(sample.scores.items())
            }
        )
        comparison_rows.append(row)

if not comparison_rows:
    raise RuntimeError(
        "No prediction produced a usable structure. Review the failure table above."
    )
comparison = pd.DataFrame(comparison_rows).sort_values(
    ["model", "seed", "sample"],
    ignore_index=True,
)
display(comparison.fillna("—"))


### Explore every predicted structure

Select any model/sample result without rerunning prediction. Chain spectrum is best for complexes; confidence coloring reads the structure's B-factor field when the writer stores per-residue confidence there.


In [ ]:
# @title 10. Interactive model and color selector
import ipywidgets as widgets
import py3Dmol
from IPython.display import clear_output

prediction_selector = widgets.Dropdown(
    options=list(STRUCTURES),
    description="Prediction:",
    layout=widgets.Layout(width="70%"),
)
color_selector = widgets.Dropdown(
    options=("Chain spectrum", "Confidence (B-factor)"),
    description="Color:",
    layout=widgets.Layout(width="45%"),
)
viewer_output = widgets.Output()


def render_structure(*_changes):
    structure_path = STRUCTURES[prediction_selector.value]
    structure_format = (
        "cif"
        if structure_path.suffix.lower() in {".cif", ".mmcif"}
        else "pdb"
    )
    with viewer_output:
        clear_output(wait=True)
        viewer = py3Dmol.view(width=900, height=560)
        viewer.addModel(structure_path.read_text(), structure_format)
        if color_selector.value == "Confidence (B-factor)":
            viewer.setStyle(
                {
                    "cartoon": {
                        "colorscheme": {
                            "prop": "b",
                            "gradient": "roygb",
                            "min": 0,
                            "max": 100,
                        }
                    }
                }
            )
        else:
            viewer.setStyle({"cartoon": {"color": "spectrum"}})
        viewer.setBackgroundColor("white")
        viewer.zoomTo()
        viewer.show()


prediction_selector.observe(render_structure, names="value")
color_selector.observe(render_structure, names="value")
display(widgets.VBox([prediction_selector, color_selector, viewer_output]))
render_structure()


### Download the comparison bundle

The archive contains the common input job, a CSV comparison table, the full batch report, and every selected model's output directory, including structures, confidence artifacts, and FoldJAX run manifests.


In [ ]:
# @title 11. Package all model outputs
import zipfile

comparison_path = WORK_DIR / f"{JOB_SLUG}-comparison.csv"
report_path = WORK_DIR / f"{JOB_SLUG}-batch-report.json"
archive_path = WORK_DIR / f"{JOB_SLUG}-foldjax-comparison.zip"
comparison.to_csv(comparison_path, index=False)
report_path.write_text(json.dumps(report.summary(), indent=2))

with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(job_path, "input/job.json")
    bundle.write(comparison_path, "comparison.csv")
    bundle.write(report_path, "batch_report.json")
    for artifact in sorted(OUTPUT_ROOT.rglob("*")):
        if artifact.is_file():
            bundle.write(
                artifact,
                Path("outputs") / artifact.relative_to(OUTPUT_ROOT),
            )

print(f"Created {archive_path} ({archive_path.stat().st_size / 1e6:.1f} MB)")
if DOWNLOAD_RESULTS:
    try:
        from google.colab import files

        files.download(str(archive_path))
    except ImportError:
        print("Download the archive from your notebook file browser.")


### Practical next steps

- Switch to **Released defaults** only after the fast workflow succeeds. The public models have different native sample/recycle defaults and released mode can take substantially longer.
- Add more seeds to inspect within-model variability. FoldJAX keeps every seed in the same reproducible result tree.
- Start with Protenix only if storage is tight. Add OpenDDE for a useful two-model comparison; add Boltz-2 when the runtime has enough disk for its larger public bundle.
- Re-running the prediction cell uses resume manifests. Re-running with a different input, model list, shape, options, weights, or runtime identity gets separate validated work.
- For ligands, DNA/RNA, templates, representations, local/gated weights, and non-Colab batch runs, see the [FoldJAX README](https://github.com/eightmm/FoldJAX).
